In [ ]:
import yaml
from jobflow import Flow
from autoplex.auto.GenMLFF.jobs import fit_mlip_ensemble
from jobflow_remote import submit_flow, set_run_config

In [ ]:
#Define resources
parallel_cpu_resources = {
    "account": "EUHPC_A04_113",
    "partition": "boost_usr_prod",
    "qos": "boost_qos_dbg",
    "time": "00:30:00",
    "nodes": 1,
    "ntasks_per_node": 32,
    "cpus_per_task": 1,
    "gres": "gpu:0",
    "mem": "480000",
    "job_name": "mlff_relax",
    "qerr_path": "mlff_relax.err",
    "qout_path": "mlff_relax.out",
}

serial_cpu_resources = {
    "account": "EUHPC_A04_113",
    "partition": "boost_usr_prod",
    "qos": "boost_qos_dbg",
    "time": "00:30:00",
    "nodes": 1,
    "ntasks_per_node": 1,
    "cpus_per_task": 32,
    "gres": "gpu:0",
    "mem": "480000",
    "job_name": "mlff_relax",
    "qerr_path": "mlff_relax.err",
    "qout_path": "mlff_relax.out",
}

serial_gpu_resources = {
    "account": "EUHPC_A04_113", 
    "partition": "boost_usr_prod",
    "qos": "boost_qos_dbg",
    "time": "00:30:00",
    "nodes": 1,
    "ntasks_per_node": 1,
    "cpus_per_task": 8,
    "gres": "gpu:1",
    "mem": "120000",
    "job_name": "mlff_relax",
    "qerr_path": "mlff_relax.err",
    "qout_path": "mlff_relax.out",
    }

In [ ]:
#Read config file
config_fname = "/leonardo_work/EUHPC_A04_113/Alberto/GenMLFF-progect/Test-workflow/rss-mlip/GenML_config.yaml"
with open(config_fname, "r") as f:
    config = yaml.safe_load(f)

ensemble_training_config = config.get("train_params", {})

In [ ]:
print("Ensemble training config:", ensemble_training_config)

In [ ]:
#Instantiate the ensemble training flow
dataset_path = "/leonardo_work/EUHPC_A04_113/Alberto/GenMLFF-progect/Test-workflow/rss-mlip/dataset/unique_dataset.extxyz"
ensemble_training = fit_mlip_ensemble(**ensemble_training_config, dataset_path=dataset_path)
flow = Flow(jobs=[ensemble_training], output=ensemble_training.output)

In [ ]:
#Set run configuration for the flow
flow = set_run_config(
    flow, name_filter="training_mlip", worker="mlff_relax_local", exec_config="mace_config", resources=serial_gpu_resources
)

In [ ]:
# Append RSSautoplex-flow to jf jobs
submit_flow(
    flow, worker="local_worker",
    resources={}, 
    project="GenMLFF",
)